In [9]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from sentence_transformers import SentenceTransformer
import faiss
import os
from tqdm import tqdm
import subprocess
import shutil
import json

In [10]:
def preprocess(s):
    if not isinstance(s, str):
        return ""
    s = s.strip()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"http\S+|www\.\S+", "<URL>", s)
    s = re.sub(r"\S+@\S+", "<EMAIL>", s)
    s = re.sub(r"<[^>]+>", " ", s)
    s = ''.join(ch for ch in s if ord(ch) >= 32)
    s = re.sub(r"['\"]", "", s)
    return s.strip()

In [18]:
BASE_DIR = Path.cwd()
IR_DIR   = BASE_DIR / "IR2025"
EMBED_DIR = BASE_DIR / "Embeddings"

DOCS_CSV   = IR_DIR / "documents.csv"
QUERIES_CSV = IR_DIR / "queries.csv"

QRELS_CSV = IR_DIR / "qrels.csv"   
QRELS_TXT = IR_DIR / "qrels.txt"   

EMBED_DIR.mkdir(exist_ok=True)
IR_DIR.mkdir(exist_ok=True)

In [12]:
df_docs = pd.read_csv(DOCS_CSV)
df_queries = pd.read_csv(QUERIES_CSV)

df_docs["Text"] = df_docs["Text"].astype(str).map(preprocess)
df_queries["Text"] = df_queries["Text"].astype(str).map(preprocess)

In [13]:
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Encoding documents...")
doc_embeddings = model.encode(
    df_docs["Text"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Encoding queries...")
query_embeddings = model.encode(
    df_queries["Text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

Encoding documents...


Batches: 100%|██████████| 287/287 [06:33<00:00,  1.37s/it]


Encoding queries...


Batches: 100%|██████████| 1/1 [00:00<00:00,  4.51it/s]


In [14]:
faiss.normalize_L2(doc_embeddings)

dim = doc_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(doc_embeddings)

print(f"FAISS index ready — vectors: {index.ntotal}, dim: {dim}")

FAISS index ready — vectors: 18316, dim: 384


In [15]:
def generate_faiss_results(df_docs, df_queries, query_embeddings, index, ks=(20, 30, 50)):
    """
    Δημιουργεί TREC runfiles για FAISS EXACTLY όπως στο Part A.
    """
    faiss.normalize_L2(query_embeddings)

    for k in ks:
        out_path = BASE_DIR / f"results_faiss_{k}.txt"
        print(f"Writing: {out_path}")

        with open(out_path, "w", encoding="utf-8") as f:
            for q_idx, row in df_queries.iterrows():
                qid = str(row["ID"])
                q_emb = query_embeddings[q_idx].reshape(1, -1)

                scores, ids = index.search(q_emb, k)

                for rank, (doc_index, score) in enumerate(zip(ids[0], scores[0]), start=1):
                    docid = str(df_docs.iloc[doc_index]["ID"])
                    f.write(f"{qid} Q0 {docid} {rank} {float(score):.4f} FAISS\n")

        print(f"✓ results_faiss_{k}.txt created successfully")


generate_faiss_results(
    df_docs=df_docs,
    df_queries=df_queries,
    query_embeddings=query_embeddings,
    index=index,
    ks=(20, 30, 50)
)


Writing: c:\Users\perik\Downloads\Semantic-Information-Retrieval-System-ElasticSearch-FAISS-Transformers-\results_faiss_20.txt
✓ results_faiss_20.txt created successfully
Writing: c:\Users\perik\Downloads\Semantic-Information-Retrieval-System-ElasticSearch-FAISS-Transformers-\results_faiss_30.txt
✓ results_faiss_30.txt created successfully
Writing: c:\Users\perik\Downloads\Semantic-Information-Retrieval-System-ElasticSearch-FAISS-Transformers-\results_faiss_50.txt
✓ results_faiss_50.txt created successfully


In [24]:
class Evaluation:

    def _rel(self, p: Path):
        try:
            rel = p.resolve().relative_to(self.data_dir.resolve())
        except Exception:
            rel = Path(p.name)
        return str(rel).replace("\\", "/")

    def __init__(self, search_client, data_dir, qrels_csv_path, qrels_txt_path, trec_eval_bin):
        self.search = search_client
        self.data_dir = Path(data_dir)
        self.qrels_csv_path = Path(qrels_csv_path)
        self.qrels_txt_path = Path(qrels_txt_path) if qrels_txt_path else self.data_dir / "qrels.txt"

        self._ensure_trec_qrels()

        if trec_eval_bin is None:
            self.trec_eval_bin = self._find_trec_eval()
        else:
            self.trec_eval_bin = Path(trec_eval_bin)

        if not self.trec_eval_bin or not self.trec_eval_bin.exists():
            raise SystemExit("trec_eval binary not found.")

        if not self.qrels_txt_path.exists():
            raise SystemExit(f"qrels file not found: {self.qrels_txt_path}")

        print(f"Using trec_eval at: {self._rel(self.trec_eval_bin)}")
        print(f"Using qrels file: {self._rel(self.qrels_txt_path)}")


    def _fix_and_write_qrels(self):
        dfq = pd.read_csv(
            self.qrels_csv_path,
            sep=None,
            engine="python",
            encoding="utf-8-sig",
        )
        dfq = dfq.dropna(how="all")
        dfq = dfq.astype(str).applymap(lambda x: x.strip())
        cols = list(dfq.columns)

        res_docids = set()
        sample_results_path = self.data_dir / "results_faiss_20.txt"
        if sample_results_path.exists():
            with open(sample_results_path, encoding="utf-8", errors="replace") as f:
                for ln in f:
                    parts = re.split(r"\s+", ln.strip())
                    if len(parts) >= 3:
                        res_docids.add(parts[2])

        candidates = []
        for c in cols:
            vals = dfq[c].dropna().astype(str).str.strip().unique()[:200].tolist()
            inter = len(set(vals) & res_docids) if res_docids else 0
            num_like = sum(1 for v in vals if re.match(r"^\d+$", v))
            candidates.append((c, inter, num_like, vals[:5]))

        docid_col = max(candidates, key=lambda x: (x[1], x[2]))[0]
        qid_col = cols[0]

        rel_col = None
        for c, inter, num_like, vals in candidates:
            if c in (qid_col, docid_col):
                continue
            sample_vals = dfq[c].dropna().astype(str).str.strip().unique()[:50]
            if sample_vals is not None and all(re.match(r"^\d+$", str(v)) for v in sample_vals):
                rel_col = c
                break
        if rel_col is None:
            others = [c for c in cols if c not in (qid_col, docid_col)]
            rel_col = others[-1] if others else cols[-1]

        with open(self.qrels_txt_path, "w", encoding="utf-8") as out:
            for _, row in dfq.iterrows():
                qid = str(row[qid_col]).strip()
                docid = str(row[docid_col]).strip()
                rel = str(row[rel_col]).strip()
                if not qid or not docid or docid.upper() == "Q0":
                    continue
                out.write(f"{qid} 0 {docid} {rel}\n")


    def _ensure_trec_qrels(self):
        if self.qrels_txt_path.exists():
            print(f"Using existing TREC qrels: {self._rel(self.qrels_txt_path)}")
            return
        print("Creating TREC qrels from CSV...")
        self._fix_and_write_qrels()


    def _find_trec_eval(self):
        candidates = [
            self.data_dir / "trec_eval" / "trec_eval.exe",
            self.data_dir / "trec_eval" / "trec_eval",
            self.data_dir / "trec_eval.exe",
            self.data_dir / "trec_eval",
        ]
        for p in candidates:
            if p.exists():
                return p

        p_on_path = shutil.which("trec_eval") or shutil.which("trec_eval.exe")
        if p_on_path:
            return Path(p_on_path)

        for p in self.data_dir.rglob("trec_eval*"):
            if p.is_file():
                return p

        return None


    def _run_trec_eval_for_file(self, results_file: Path):
        if not results_file.exists():
            raise SystemExit(f"results file not found: {results_file}")

        cmd = (
            f"{self._rel(self.trec_eval_bin)} "
            f"{self._rel(self.qrels_txt_path)} "
            f"{self._rel(results_file)} -m all_trec"
        )

        print("Running:", cmd)

        proc = subprocess.run(cmd, capture_output=True, text=True)

        if proc.returncode != 0:
            print("--- trec_eval stderr ---")
            print(proc.stderr)
            raise SystemExit(f"trec_eval failed (rc={proc.returncode})")

        parsed = {}

        for line in proc.stdout.strip().splitlines():
            parts = re.split(r"\s+", line.strip())
            if len(parts) < 3:
                continue

            metric = parts[0]
            target = parts[1]
            value_str = parts[-1]

            if target.lower() != "all":
                continue

            if metric.lower() == "map":
                parsed["MAP"] = float(value_str)
                continue

            m = re.match(r"^P_(\d+)$", metric)
            if m:
                k = int(m.group(1))
                parsed[f"P@{k}"] = float(value_str)

        return {
            "P@5":  parsed.get("P@5"),
            "P@10": parsed.get("P@10"),
            "P@15": parsed.get("P@15"),
            "P@20": parsed.get("P@20"),
            "MAP":  parsed.get("MAP"),
        }


    def evaluate_with_trec_eval(self, ks=(20,30,50), out_summary_path=None):
        summary_rows = []

        for k in ks:
            results_file = self.data_dir / f"results_faiss_{k}.txt"
            metrics = self._run_trec_eval_for_file(results_file)
            row = {"retrieval_k": k}
            row.update(metrics)
            summary_rows.append(row)

        df_trec = pd.DataFrame(summary_rows).sort_values("retrieval_k").reset_index(drop=True)

        print("\nTREC Evaluation Summary:")
        print(df_trec)

        if out_summary_path is not None:
            with open(out_summary_path, "w", encoding="utf-8") as f:
                for _, r in df_trec.iterrows():
                    k = int(r["retrieval_k"])
                    f.write(f"# summary for results_faiss_{k}.txt\n")
                    for kk in [5,10,15,20]:
                        val = r.get(f"P@{kk}")
                        f.write(f"P_{kk}\tall\t{val:.4f}\n")
                    f.write(f"map\tall\t{r['MAP']:.4f}\n\n")

        return df_trec




In [29]:
evaluator = Evaluation(
    search_client=None,
    data_dir=BASE_DIR,
    qrels_csv_path=QRELS_CSV,
    qrels_txt_path=QRELS_TXT,
    trec_eval_bin=IR_DIR / "trec_eval" / "trec_eval.exe"
)

def evaluate_faiss(evaluator, ks=(20,30,50)):
    summary_rows = []

    for k in ks:
        # Πρέπει να είναι PATH, όχι string
        results_file = Path(f"results_faiss_{k}.txt")

        metrics = evaluator._run_trec_eval_for_file(results_file)

        row = {"retrieval_k": k}
        row.update(metrics)
        summary_rows.append(row)

    df = (
        pd.DataFrame(summary_rows)
        .sort_values("retrieval_k")
        .reset_index(drop=True)
    )

    print("\nFAISS Evaluation Summary:")
    print(df)

    return df



print("\nDONE — FAISS Part B completed successfully!")

Using existing TREC qrels: IR2025/qrels.txt
Using trec_eval at: IR2025/trec_eval/trec_eval.exe
Using qrels file: IR2025/qrels.txt

DONE — FAISS Part B completed successfully!


In [ ]:
df_faiss = evaluate_faiss(evaluator, ks=(20,30,50))

Running: IR2025/trec_eval/trec_eval.exe IR2025/qrels.txt results_faiss_20.txt -m all_trec
Running: IR2025/trec_eval/trec_eval.exe IR2025/qrels.txt results_faiss_30.txt -m all_trec
Running: IR2025/trec_eval/trec_eval.exe IR2025/qrels.txt results_faiss_50.txt -m all_trec

FAISS Evaluation Summary:
   retrieval_k   P@5  P@10    P@15   P@20     MAP
0           20  0.66  0.48  0.3733  0.355  0.3146
1           30  0.66  0.48  0.3733  0.355  0.3415
2           50  0.66  0.48  0.3733  0.355  0.3759
